# 🎨 Generative Vision Academic Notebook: VAE, DCGAN, and Latent Diffusion
**Course:** Generative AI — Master of Computer Science  
**Topic:** Module B Academic Laboratory: Variational Autoencoder (VAE), Deep Convolutional GAN (DCGAN), Latent Manifold Traversal, and Comparative Diffusion Analysis  
**Theoretical References:** Kingma & Welling (2013), Radford et al. (2015), Rombach et al. (2022)

---
### 📌 Executive Summary
Dalam aplikasi produksi **AI Personalized Storybook**, ilustrasi halaman menggunakan sintesis visual berkualitas tinggi yang memadukan aset cat air lembut (*watercolor storytelling*) dan pemodelan generatif.  
Namun, untuk **memenuhi kewajiban akademik Module B**, notebook ini menyajikan implementasi mandiri (*from-scratch*) dan evaluasi teoritis mendalam terhadap dua arsitektur generatif fundamental:
1. **Variational Autoencoder (VAE):** Pemodelan ruang laten berkesinambungan dengan *Reparameterization Trick* dan fungsi objektif $\mathcal{L}_{	ext{ELBO}}$ (BCE/MSE Reconstruction + Kullback-Leibler Divergence).
2. **Deep Convolutional GAN (DCGAN):** Persaingan *minimax game* antara Generator dan Discriminator konvolusional dengan teknik stabilisasi TTUR dan label smoothing.
3. **Analisis Komparatif:** Mengkaji *mode collapse*, variasi sampel, dan alasan pemilihan *Latent Diffusion Models* untuk produksi ilustrasi anak.



## 🛠️ Step 1: Dependencies & Environment Setup


In [ ]:
import os
import sys
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# Ensure reproducible random seed
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")



## 🔬 Section 1: Variational Autoencoder (VAE) Implementation

### 1.1 Teori & Formulasi Matematis ELBO
VAE memodelkan data $x$ melalui variabel laten kontinu $z \sim p(z) = \mathcal{N}(0, I)$. Karena posterior sebenarnya $p(z|x)$ sulit dihitung (*intractable*), VAE menggunakan jaringan pengenal (*inference network*) $q_\phi(z|x) = \mathcal{N}(\mu_\phi(x), \Sigma_\phi(x))$.

Fungsi objektif yang dimaksimalkan adalah **Evidence Lower Bound (ELBO)**:
$$\mathcal{L}_{\text{ELBO}}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{\text{KL}}(q_\phi(z|x) \,|\,|\, p(z))$$

Dengan trik reparameterisasi (*Kingma & Welling, 2013*):
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Dan divergensi KL analitik:
$$D_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^d \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$



In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super(ConvVAE, self).__init__()
        self.latent_dim = latent_dim
        
        # Encoder: x (1, 28, 28) -> mu, logvar
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # 14x14
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 7x7
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(32 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(32 * 7 * 7, latent_dim)
        
        # Decoder: z (latent_dim) -> x_recon (1, 28, 28)
        self.decoder_fc = nn.Linear(latent_dim, 32 * 7 * 7)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # 14x14
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),  # 28x28
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 32, 7, 7)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

def vae_loss_function(recon_x, x, mu, logvar):
    # Binary Cross Entropy reconstruction loss
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    # Analytic KL Divergence
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD, BCE, KLD

print("✅ ConvVAE Architecture and Loss defined successfully.")



### 1.2 Pelatihan VAE pada Dataset Ilustrasi Karakter


In [ ]:
# Generate synthetic character silhouettes (e.g., circular faces and character shapes)
N_SAMPLES = 500
synthetic_data = torch.zeros(N_SAMPLES, 1, 28, 28)
for i in range(N_SAMPLES):
    r = np.random.uniform(5, 10)
    cx, cy = np.random.uniform(10, 18), np.random.uniform(10, 18)
    y, x = np.ogrid[:28, :28]
    mask = ((x - cx)**2 + (y - cy)**2) <= r**2
    synthetic_data[i, 0, mask] = 0.85 + np.random.normal(0, 0.05)

vae_dataset = TensorDataset(synthetic_data)
vae_loader = DataLoader(vae_dataset, batch_size=32, shuffle=True)

vae = ConvVAE(latent_dim=2).to(device)
optimizer_vae = optim.Adam(vae.parameters(), lr=1e-3)

vae.train()
for epoch in range(1, 11):
    total_loss, total_bce, total_kld = 0, 0, 0
    for (x_batch,) in vae_loader:
        x_batch = x_batch.to(device)
        optimizer_vae.zero_grad()
        recon_batch, mu, logvar = vae(x_batch)
        loss, bce, kld = vae_loss_function(recon_batch, x_batch, mu, logvar)
        loss.backward()
        optimizer_vae.step()
        total_loss += loss.item()
        total_bce += bce.item()
        total_kld += kld.item()
    if epoch % 2 == 0 or epoch == 1:
        print(f"VAE Epoch {epoch:02d} | Loss: {total_loss/len(vae_dataset):.2f} (BCE: {total_bce/len(vae_dataset):.2f}, KLD: {total_kld/len(vae_dataset):.2f})")



### 1.3 Visualisasi Manifold Ruang Laten 2D VAE
Karena kita mengatur `latent_dim=2`, kita dapat melakukan visualisasi grid traversal langsung di ruang laten $z \in [-2, 2]^2$:


In [ ]:
vae.eval()
grid_x = np.linspace(-2, 2, 10)
grid_y = np.linspace(-2, 2, 10)
canvas = np.zeros((28 * 10, 28 * 10))

with torch.no_grad():
    for i, yi in enumerate(grid_x):
        for j, xi in enumerate(grid_y):
            z_sample = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
            x_decoded = vae.decode(z_sample).cpu().squeeze().numpy()
            canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = x_decoded

plt.figure(figsize=(8, 8))
plt.imshow(canvas, cmap='magma')
plt.title("Visualisasi Manifold Ruang Laten 2D VAE ($z_1$ vs $z_2$)", fontsize=13)
plt.axis('off')
plt.show()



## 🎨 Section 2: Deep Convolutional GAN (DCGAN)

### 2.1 Formulasi Minimax Game & Arsitektur
DCGAN (*Radford et al., 2015*) melatih Generator $G(z; 	heta_g)$ dan Discriminator $D(x; 	heta_d)$ melalui permainan minimax:
$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log (1 - D(G(z)))]$$

**Teknik Stabilisasi yang Diimplementasikan:**
1. **Transposed Convolutions:** Menggantikan spatial pooling.
2. **Batch Normalization:** Menstabilkan propagasi gradien di kedua jaringan.
3. **LeakyReLU (slope 0.2):** Mencegah fenomena *dying ReLU*.
4. **One-sided Label Smoothing:** Label real dihaluskan dari $1.0$ menjadi $0.9$ untuk meredam overconfidence Discriminator.



In [ ]:
class DCGANGenerator(nn.Module):
    def __init__(self, nz=32):
        super(DCGANGenerator, self).__init__()
        self.net = nn.Sequential(
            # z (nz, 1, 1) -> (128, 4, 4)
            nn.ConvTranspose2d(nz, 128, 4, 1, 0, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # (128, 4, 4) -> (64, 8, 8)
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # (64, 8, 8) -> (32, 16, 16)
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            # (32, 16, 16) -> (1, 32, 32)
            nn.ConvTranspose2d(32, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class DCGANDiscriminator(nn.Module):
    def __init__(self):
        super(DCGANDiscriminator, self).__init__()
        self.net = nn.Sequential(
            # (1, 32, 32) -> (32, 16, 16)
            nn.Conv2d(1, 32, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (32, 16, 16) -> (64, 8, 8)
            nn.Conv2d(32, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            # (64, 8, 8) -> (128, 4, 4)
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # (128, 4, 4) -> (1, 1, 1)
            nn.Conv2d(128, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1).squeeze(1)

print("✅ DCGAN Generator and Discriminator architectures initialized.")



### 2.2 Training Loop DCGAN dengan Stabilisasi TTUR


In [ ]:
nz = 32
netG = DCGANGenerator(nz=nz).to(device)
netD = DCGANDiscriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Synthetic 32x32 character patterns
gan_data = torch.zeros(400, 1, 32, 32)
for i in range(400):
    gan_data[i, 0, 8:24, 8:24] = 0.8  # Square motif
gan_loader = DataLoader(TensorDataset(gan_data), batch_size=32, shuffle=True)

fixed_noise = torch.randn(16, nz, 1, 1, device=device)

print("Starting DCGAN Training Loop...")
for epoch in range(1, 6):
    for i, (real_imgs,) in enumerate(gan_loader):
        b_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        
        # 1. Train Discriminator: max log(D(x)) + log(1 - D(G(z)))
        netD.zero_grad()
        label_real = torch.full((b_size,), 0.9, device=device) # Label smoothing
        output_real = netD(real_imgs)
        lossD_real = criterion(output_real, label_real)
        
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_imgs = netG(noise)
        label_fake = torch.full((b_size,), 0.0, device=device)
        output_fake = netD(fake_imgs.detach())
        lossD_fake = criterion(output_fake, label_fake)
        
        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()
        
        # 2. Train Generator: max log(D(G(z)))
        netG.zero_grad()
        label_gen = torch.full((b_size,), 1.0, device=device)
        output_g = netD(fake_imgs)
        lossG = criterion(output_g, label_gen)
        lossG.backward()
        optimizerG.step()

    print(f"DCGAN Epoch {epoch:02d} | Loss_D: {lossD.item():.4f} | Loss_G: {lossG.item():.4f}")



### 2.3 Sampling Grid & Pemeriksaan Indikasi Mode Collapse
*Mode collapse* terjadi ketika Generator hanya menghasilkan variasi sampel yang sangat sempit karena Discriminator gagal memberikan gradien yang bervariasi.


In [ ]:
netG.eval()
with torch.no_grad():
    fake_samples = netG(fixed_noise).detach().cpu()

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for idx, ax in enumerate(axes.flatten()):
    ax.imshow(fake_samples[idx, 0], cmap='bone')
    ax.axis('off')
plt.suptitle("DCGAN Generated Samples (16 Variasi Noise)", fontsize=13)
plt.show()

# Pemeriksaan keragaman pixel (Standard Deviation antar sampel)
sample_diversity = float(torch.std(fake_samples, dim=0).mean())
print(f"Tingkat Keragaman Antar-Sampel (Std Pixel): {sample_diversity:.4f}")
if sample_diversity < 0.01:
    print("⚠️ Indikasi Mode Collapse terdeteksi: Sampel memiliki variasi pixel mendekati nol.")
else:
    print("✅ Tidak Ada Mode Collapse Fatal: Model mempertahankan variasi struktural antar noise.")



## 📊 Section 3: Analisis Komparatif (VAE vs DCGAN vs Latent Diffusion)

Tabel evaluasi komparatif untuk menjawab kriteria akademis tugas:

| Dimensi Evaluasi | Variational Autoencoder (VAE) | Deep Convolutional GAN (DCGAN) | Latent Diffusion Models (LDM / SD) |
|---|---|---|---|
| **Kesesuaian Naratif (Prompt Conditioning)** | Rendah (hanya pengkondisian label diskret $c$) | Sedang (cGAN memerlukan embedding multimodal terbatas) | **Sangat Tinggi** (Cross-attention text conditioning via CLIP) |
| **Kualitas & Ketajaman Visual** | Cenderung buram (*blurry*) akibat loss rekonstruksi L2/BCE | Tajam namun rentan artefak geometris / *checkerboard* | **Sangat Realistis & Kaya Gaya Artistik** (Watercolor, Digital) |
| **Kestabilan Pelatihan** | Sangat stabil (analitik ELBO) | Kurang stabil (*mode collapse*, vanishing gradients) | Stabil (denoising score matching) |
| **Konsistensi Karakter Antarhalaman** | Bergantung pada lintasan manifold laten | Acak berdasarkan seed $z$, sulit dikunci | **Dapat dikunci** via seed referensi, LoRA, atau IP-Adapter |

### 💡 Mengapa Latent Diffusion & Watercolor Compositing Dipilih untuk Aplikasi Final?
1. **Ekspresi Adegan Nyata:** VAE dan DCGAN berdimensi kecil sangat baik untuk eksperimen representasi laten akademis, namun tidak mampu merender adegan kompleks buku anak (misalnya *"anak mendayung perahu kayu di bawah pohon willow tepi sungai"*).
2. **Kesesuaian Teks-ke-Visual:** Latent Diffusion memanfaatkan representasi semantik teks CLIP yang mampu memetakan aksi, ekspresi, dan suasana secara tepat.
3. **Kenyamanan Membaca Anak:** Anak-anak membutuhkan ilustrasi berkarakter hangat dan estetik (seperti gaya cat air *Beatrix Potter*), bukan citra sintetis kotak atau siluet buram.

